<a href="https://colab.research.google.com/github/markoutsikou/DWS101-ML/blob/main/ML_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Library installation - input**

In [16]:
!pip -q install catboost lightgbm xgboost shap
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
import zipfile

# **Data load**

In [5]:
train_feat = pd.read_csv("train_hh_features.csv")
test_feat  = pd.read_csv("test_hh_features.csv")
train_y    = pd.read_csv("train_hh_gt.csv")
train_rates= pd.read_csv("train_rates_gt.csv")

train = train_feat.merge(train_y, on=["survey_id", "hhid"], how="left")

print("train_feat:", train_feat.shape)
print("train_y   :", train_y.shape)
print("train     :", train.shape)
print("test_feat :", test_feat.shape)

train.head()

train_feat: (104234, 88)
train_y   : (104234, 3)
train     : (104234, 89)
test_feat : (103023, 88)


,hhid,com,weight,strata,utl_exp_ppp17,male,hsize,num_children5,num_children10,num_children18,...,consumed4300,consumed4400,consumed4500,consumed4600,consumed4700,consumed4800,consumed4900,consumed5000,survey_id,cons_ppp17
0,100001,1,75,4,594.80627,Female,1,0,0,0,...,No,No,No,Yes,Yes,Yes,Yes,No,100000,25.258402
1,100002,1,150,4,1676.27230,Female,2,0,0,0,...,No,No,No,No,Yes,Yes,No,No,100000,16.996706
2,100003,1,375,4,506.93719,Male,5,0,0,2,...,Yes,No,Yes,Yes,Yes,Yes,No,Yes,100000,13.671848
3,100004,1,375,4,824.61786,Male,5,0,0,1,...,Yes,No,No,No,Yes,Yes,No,No,100000,7.189475
4,100005,1,525,4,351.47644,Male,7,1,0,0,...,No,No,Yes,No,Yes,Yes,Yes,No,100000,12.308855


# **Target, thresholds, weight column**

In [6]:
target_col = "cons_ppp17"
weight_col = "weight"

rate_cols = [c for c in train_rates.columns if c != "survey_id"]
thresholds = np.array([float(c.replace("pct_hh_below_","")) for c in rate_cols])

print("TARGET:", target_col)
print("WEIGHT:", weight_col)
print("N thresholds:", len(thresholds))
print("First 5 thresholds:", thresholds[:5])


TARGET: cons_ppp17
WEIGHT: weight
N thresholds: 19
First 5 thresholds: [3.17 3.94 4.6  5.26 5.88]


# **Features/IDs**

In [7]:
id_cols = ["survey_id", "hhid"]

X = train.drop(columns=id_cols + [target_col])
y = train[target_col].astype(float)

X_test = test_feat.drop(columns=id_cols)

print("X:", X.shape, "y:", y.shape, "X_test:", X_test.shape)

X: (104234, 86) y: (104234,) X_test: (103023, 86)


# **Categorical columns**

In [9]:
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
cat_idx  = [X.columns.get_loc(c) for c in cat_cols]

print("Categorical cols:", len(cat_cols))
print("Example:", cat_cols[:10])

Categorical cols: 64
Example: ['male', 'owner', 'water', 'toilet', 'sewer', 'elect', 'water_source', 'sanitation_source', 'dweltyp', 'employed']


# **log-target + GroupKFold by survey**

In [10]:
y_log = np.log1p(y)
groups = train["survey_id"].values
gkf = GroupKFold(n_splits=3)

# **Poverty rates + blended metric**

In [11]:
def weighted_poverty_rates(y_pred, w, thresholds):
    w = np.asarray(w).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    denom = w.sum()
    rates = []
    for t in thresholds:
        rates.append((w * (y_pred < t)).sum() / denom)
    return np.array(rates)

def blended_metric_for_one_survey(y_true, y_pred, w, true_rates_row, rate_cols, thresholds):
    cons_mape = np.mean(np.abs((y_pred - y_true) / np.maximum(y_true, 1e-9)))

    true_rates = true_rates_row[rate_cols].values.astype(float)
    pred_rates = weighted_poverty_rates(y_pred, w, thresholds)
    rate_mape_each = np.abs((pred_rates - true_rates) / np.maximum(true_rates, 1e-9))

    k = np.arange(len(thresholds))
    center = 8
    weights = np.exp(-0.5*((k-center)/3.0)**2)
    weights = weights / weights.sum()

    rates_mape = np.sum(weights * rate_mape_each)
    blended = 0.9 * rates_mape + 0.1 * cons_mape
    return blended, cons_mape, rates_mape


# **Managing NaNs**

In [14]:
obj_cols = [c for c in X.columns if X[c].dtype == "object"]
print("Object columns:", obj_cols)

if len(obj_cols) > 0:
    nan_counts = X[obj_cols].isna().sum().sort_values(ascending=False)
    print("\nNaN counts in object cols:")
    print(nan_counts[nan_counts > 0].head(20))


cat_cols = [c for c in X.columns if X[c].dtype == "object"]
for c in cat_cols:
    X[c] = X[c].fillna("__MISSING__").astype(str)
    X_test[c] = X_test[c].fillna("__MISSING__").astype(str)

cat_idx = [X.columns.get_loc(c) for c in cat_cols]

print("Fixed categoricals:", len(cat_cols))
print("Example:", cat_cols[:10])
num_cols = [c for c in X.columns if c not in cat_cols]
med = X[num_cols].median()

X[num_cols] = X[num_cols].fillna(med)
X_test[num_cols] = X_test[num_cols].fillna(med)

print("Numeric imputation done.")


Object columns: ['male', 'owner', 'water', 'toilet', 'sewer', 'elect', 'water_source', 'sanitation_source', 'dweltyp', 'employed', 'educ_max', 'any_nonagric', 'sector1d', 'urban', 'consumed100', 'consumed200', 'consumed300', 'consumed400', 'consumed500', 'consumed600', 'consumed700', 'consumed800', 'consumed900', 'consumed1000', 'consumed1100', 'consumed1200', 'consumed1300', 'consumed1400', 'consumed1500', 'consumed1600', 'consumed1700', 'consumed1800', 'consumed1900', 'consumed2000', 'consumed2100', 'consumed2200', 'consumed2300', 'consumed2400', 'consumed2500', 'consumed2600', 'consumed2700', 'consumed2800', 'consumed2900', 'consumed3000', 'consumed3100', 'consumed3200', 'consumed3300', 'consumed3400', 'consumed3500', 'consumed3600', 'consumed3700', 'consumed3800', 'consumed3900', 'consumed4000', 'consumed4100', 'consumed4200', 'consumed4300', 'consumed4400', 'consumed4500', 'consumed4600', 'consumed4700', 'consumed4800', 'consumed4900', 'consumed5000']

NaN counts in object cols:
s

# **LOSOCV training με CatBoost**

In [17]:
oof_pred = np.zeros(len(train), dtype=float)
scores = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y_log, groups=groups), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr = y_log.iloc[tr_idx]
    y_va_true = y.iloc[va_idx]
    w_tr = train.iloc[tr_idx][weight_col].values
    w_va = train.iloc[va_idx][weight_col].values

    model = CatBoostRegressor(
        loss_function="RMSE",
        depth=6,
        learning_rate=0.1,
        iterations=1500,
        random_seed=42,
        verbose=200,
        od_type="Iter",
        od_wait=80,
        thread_count=-1
    )

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_idx, weight=w_tr)
    va_pool = Pool(X_va, np.log1p(y_va_true), cat_features=cat_idx)

    model.fit(tr_pool, eval_set=va_pool, use_best_model=True)

    pred = np.expm1(model.predict(X_va))
    pred = np.clip(pred, 0, None)
    oof_pred[va_idx] = pred

    survey_id = train.iloc[va_idx]["survey_id"].iloc[0]
    true_rates_row = train_rates[train_rates["survey_id"] == survey_id].iloc[0]

    blended, cons_mape, rates_mape = blended_metric_for_one_survey(
        y_true=y_va_true.values,
        y_pred=pred,
        w=w_va,
        true_rates_row=true_rates_row,
        rate_cols=rate_cols,
        thresholds=thresholds
    )

    scores.append((survey_id, blended, cons_mape, rates_mape, model.get_best_iteration()))
    print(f"Fold {fold} | survey {survey_id} | blended={blended:.6f} cons_mape={cons_mape:.6f} rates_mape={rates_mape:.6f} best_iter={model.get_best_iteration()}")

scores_df = pd.DataFrame(scores, columns=["survey_id","blended","cons_mape","rates_mape","best_iter"])
print("\nMEAN blended:", scores_df["blended"].mean())
scores_df

0:	learn: 0.5680358	test: 0.5987129	best: 0.5987129 (0)	total: 630ms	remaining: 15m 44s
200:	learn: 0.2940035	test: 0.3201202	best: 0.3201202 (200)	total: 1m 31s	remaining: 9m 53s
400:	learn: 0.2730025	test: 0.3104872	best: 0.3104872 (400)	total: 3m 23s	remaining: 9m 17s
600:	learn: 0.2648214	test: 0.3085284	best: 0.3085284 (600)	total: 5m 15s	remaining: 7m 51s
800:	learn: 0.2590722	test: 0.3079637	best: 0.3079616 (796)	total: 7m 6s	remaining: 6m 11s
1000:	learn: 0.2543861	test: 0.3075023	best: 0.3074987 (999)	total: 8m 56s	remaining: 4m 27s
1200:	learn: 0.2507923	test: 0.3072860	best: 0.3072699 (1158)	total: 10m 46s	remaining: 2m 40s
1400:	learn: 0.2471449	test: 0.3070109	best: 0.3069784 (1359)	total: 12m 37s	remaining: 53.5s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.3069784301
bestIteration = 1359

Shrink model to first 1360 iterations.
Fold 1 | survey 300000 | blended=0.072974 cons_mape=0.278434 rates_mape=0.050146 best_iter=1359
0:	learn: 0.5707164	test: 0

,survey_id,blended,cons_mape,rates_mape,best_iter
0,300000,0.072974,0.278434,0.050146,1359
1,200000,0.062655,0.276769,0.038864,1498
2,100000,0.081917,0.284317,0.059428,1472


# **Train final model on all train + predict test**

In [18]:
final_model = CatBoostRegressor(
    loss_function="RMSE",
    depth=8,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200,
    thread_count=-1
)

all_pool = Pool(
    X,
    y_log,
    cat_features=cat_idx,
    weight=train[weight_col].values
)

final_model.fit(all_pool)

0:	learn: 0.5795251	total: 773ms	remaining: 25m 45s
200:	learn: 0.3385156	total: 2m 40s	remaining: 23m 56s
400:	learn: 0.2856030	total: 6m 29s	remaining: 25m 51s
600:	learn: 0.2703236	total: 10m 59s	remaining: 25m 35s
800:	learn: 0.2616890	total: 15m 31s	remaining: 23m 14s
1000:	learn: 0.2554886	total: 20m 4s	remaining: 20m 1s
1200:	learn: 0.2503903	total: 24m 32s	remaining: 16m 19s
1400:	learn: 0.2465637	total: 28m 54s	remaining: 12m 21s
1600:	learn: 0.2435641	total: 33m 7s	remaining: 8m 15s
1800:	learn: 0.2407503	total: 37m 20s	remaining: 4m 7s
1999:	learn: 0.2379698	total: 41m 38s	remaining: 0us


# **Predict on test set**

In [19]:
test_pred_log = final_model.predict(X_test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)

print("Test predictions summary:")
print("min:", test_pred.min(), "median:", np.median(test_pred), "max:", test_pred.max())


Test predictions summary:
min: 1.4191946920073049 median: 9.514787940163147 max: 122.27208353319548


# **Create CSVs and submission.zip**

In [25]:
pred_household = test_feat[["survey_id","hhid"]].copy()
pred_household["cons_ppp17"] = test_pred
pred_household.to_csv("predicted_household_consumption.csv", index=False)
print("Wrote predicted_household_consumption.csv", pred_household.shape)

poverty_rows = []
for sid in sorted(test_feat["survey_id"].unique()):
    mask = (test_feat["survey_id"] == sid)
    w = test_feat.loc[mask, weight_col].values
    yhat = pred_household.loc[mask, "cons_ppp17"].values

    rates = weighted_poverty_rates(yhat, w, thresholds)
    row = {"survey_id": sid}
    for c, r in zip(rate_cols, rates):
        row[c] = r
    poverty_rows.append(row)

pred_rates_df = pd.DataFrame(poverty_rows, columns=["survey_id"] + rate_cols)
pred_rates_df.to_csv("predicted_poverty_distribution.csv", index=False)
print("Wrote predicted_poverty_distribution.csv", pred_rates_df.shape)

with zipfile.ZipFile("submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write("predicted_household_consumption.csv")
    z.write("predicted_poverty_distribution.csv")

print("Created submission.zip")

Wrote predicted_household_consumption.csv (103023, 3)
Wrote predicted_poverty_distribution.csv (3, 20)
Created submission.zip
